# Full Train: Question-Gated GPA + RepViT-M1.5 + T5-Efficient-Mini on Kaggle T4 x2

Notebook này dùng safe config cố định cho experiment đầu tiên: `model.gpa_conditioning: question_gate`. Luồng vẫn giữ cache visual features: `prepare-data` tạo JSONL, `prepare-features` tạo memmap, train/eval đọc lại feature cache nên không chạy lại frozen visual encoder trong lúc train nếu cache/manifest khớp.

Trước khi chạy trên Kaggle, hãy chắc chắn branch bạn clone đã có các file config question-GPA mới.

In [ ]:
GITHUB_REPO_URL = "https://github.com/huyvanzzz/Group_HND_car.git"
GITHUB_BRANCH = "nguyet"
RUN_PROGRESS_MONITOR = False
EVAL_WITH_METEOR = False

import os
os.environ["GITHUB_REPO_URL"] = GITHUB_REPO_URL
os.environ["GITHUB_BRANCH"] = GITHUB_BRANCH
os.environ["RUN_PROGRESS_MONITOR"] = "1" if RUN_PROGRESS_MONITOR else "0"
os.environ["EVAL_WITH_METEOR"] = "1" if EVAL_WITH_METEOR else "0"
os.environ["QUESTION_GPA_CONFIG"] = "configs/repvit_t5_efficient_mini_question_gpa_kaggle_2gpu_safe.yaml"
os.environ["QUESTION_GPA_PROFILE"] = "outputs/repvit_t5_efficient_mini_question_gpa_kaggle_2gpu_safe"

print({
    "repo": GITHUB_REPO_URL,
    "branch": GITHUB_BRANCH,
    "config": os.environ["QUESTION_GPA_CONFIG"],
    "profile": os.environ["QUESTION_GPA_PROFILE"],
    "progress_monitor": RUN_PROGRESS_MONITOR,
    "eval_with_meteor": EVAL_WITH_METEOR,
})

## 1. Clone repo và cài package

Cell này clone/pull repo từ branch bạn chọn, rồi cài package ở editable mode kèm dev dependency để có `pytest` cho smoke tests.

In [ ]:
%%bash
set -e
cd /kaggle/working
if [ ! -d Efficient_VLM_For_Autonomous_Driving ]; then
  git clone --branch "$GITHUB_BRANCH" "$GITHUB_REPO_URL" Efficient_VLM_For_Autonomous_Driving
fi
cd Efficient_VLM_For_Autonomous_Driving
git pull --ff-only
python -m pip install -e ".[dev]"

## 2. Token và GPU check

In [ ]:
from kaggle_secrets import UserSecretsClient
import os
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
print({"hf_token_present": bool(os.environ.get("HF_TOKEN"))})

In [ ]:
import torch
count = torch.cuda.device_count()
print({"cuda": torch.cuda.is_available(), "gpu_count": count})
for idx in range(count):
    print(idx, torch.cuda.get_device_name(idx))
if count < 2:
    raise RuntimeError("Full train notebook requires Kaggle GPU T4 x2.")

In [ ]:
%%bash
set -e
nvidia-smi
ps -ef | grep -E "accelerate|efficient_vlm_ad|python" | grep -v grep || true

## 3. Sanity check config

Cell này kiểm tra đúng config `question_gate` và đúng safe batch setup trước khi chạy prepare/train.

In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
CONFIG="$QUESTION_GPA_CONFIG"
python -m efficient_vlm_ad inspect-data --config "$CONFIG"
python - <<'PY'
import os
from pathlib import Path
import yaml

config_path = Path(os.environ["QUESTION_GPA_CONFIG"])
cfg = yaml.safe_load(config_path.read_text())
summary = {
    "config": str(config_path),
    "output_dir": cfg["project"]["output_dir"],
    "profile": cfg["model"]["profile"],
    "gpa_conditioning": cfg["model"].get("gpa_conditioning"),
    "batch_size_per_gpu": cfg["training"]["batch_size"],
    "gradient_accumulation_steps": cfg["training"]["gradient_accumulation_steps"],
    "effective_batch_2gpu": cfg["training"]["batch_size"] * cfg["training"]["gradient_accumulation_steps"] * 2,
}
print(summary)
assert cfg["model"].get("gpa_conditioning") == "question_gate"
assert cfg["training"]["batch_size"] == 8
assert cfg["training"]["gradient_accumulation_steps"] == 4
PY

## 4. Prepare data và cache frozen visual features

`prepare-features` là bước tốn thời gian chạy image encoder. Sau khi cache tạo xong, train sẽ đọc `features.float16.memmap` thay vì encode lại ảnh.

In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
CONFIG="$QUESTION_GPA_CONFIG"
python -m efficient_vlm_ad prepare-data --config "$CONFIG" --subset full
python -m efficient_vlm_ad prepare-features --config "$CONFIG" --subset full

## 5. Focused tests và 1-step train probe

Cell này chạy test/config/modeling nhanh nếu môi trường Kaggle cài đủ dependency, sau đó chạy diagnose và 1-step 2GPU probe cho stage align. Probe checkpoint sẽ bị xóa để full train bắt đầu sạch.

In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
CONFIG="$QUESTION_GPA_CONFIG"
PROFILE="$QUESTION_GPA_PROFILE"

python -m pytest tests/test_config.py tests/test_modeling.py -q
python -m efficient_vlm_ad diagnose-train --config "$CONFIG" --stage align --debug --debug-samples 1 --debug-numerics

if [ -f "$PROFILE/checkpoints/align_latest.pt" ]; then
  echo "Skipping 1-step 2GPU probe because align_latest.pt already exists. Delete align checkpoints before probing from scratch."
else
  PYTHONUNBUFFERED=1 accelerate launch --multi_gpu --num_processes 2 --num_machines 1 --mixed_precision no --dynamo_backend no -m efficient_vlm_ad train --config "$CONFIG" --stage align --max-steps 1 --debug --debug-samples 1 --debug-numerics
  rm -f "$PROFILE/checkpoints/align_latest.pt" "$PROFILE/checkpoints/align_best.pt"
fi

## 6. Train stage align

Nếu đã có `align_latest.pt`, cell này tự resume. Nếu chưa có, nó bắt đầu align từ đầu.

In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
CONFIG="$QUESTION_GPA_CONFIG"
PROFILE="$QUESTION_GPA_PROFILE"
RESUME=""
if [ -f "$PROFILE/checkpoints/align_latest.pt" ]; then
  RESUME="--resume $PROFILE/checkpoints/align_latest.pt"
fi
PYTHONUNBUFFERED=1 accelerate launch --multi_gpu --num_processes 2 --num_machines 1 --mixed_precision no --dynamo_backend no -m efficient_vlm_ad train --config "$CONFIG" --stage align $RESUME --debug --debug-samples 1 --debug-numerics

In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
PROFILE="$QUESTION_GPA_PROFILE"
tail -n 20 "$PROFILE/debug/train_progress.jsonl" || true
python -m efficient_vlm_ad monitor-progress --output-dir "$PROFILE" --last 5 || true

In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
PROFILE="$QUESTION_GPA_PROFILE"
if [ "$RUN_PROGRESS_MONITOR" = "1" ]; then
  while true; do clear; date; tail -n 5 "$PROFILE/debug/train_progress.jsonl" || true; sleep 30; done
else
  tail -n 5 "$PROFILE/debug/train_progress.jsonl" || true
fi

## 7. Train stage finetune

Nếu có `finetune_latest.pt`, cell này resume finetune. Nếu chưa có, nó bắt đầu finetune từ `align_best.pt`.

In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
CONFIG="$QUESTION_GPA_CONFIG"
PROFILE="$QUESTION_GPA_PROFILE"

if [ -f "$PROFILE/checkpoints/finetune_latest.pt" ]; then
  RESUME="$PROFILE/checkpoints/finetune_latest.pt"
else
  RESUME="$PROFILE/checkpoints/align_best.pt"
fi

if [ ! -f "$RESUME" ]; then
  echo "Missing resume checkpoint: $RESUME"
  echo "Run align first, or upload/copy the expected checkpoint."
  exit 1
fi

PYTHONUNBUFFERED=1 accelerate launch --multi_gpu --num_processes 2 --num_machines 1 --mixed_precision no --dynamo_backend no -m efficient_vlm_ad train --config "$CONFIG" --stage finetune --resume "$RESUME" --debug --debug-samples 1 --debug-numerics

In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
PROFILE="$QUESTION_GPA_PROFILE"
tail -n 20 "$PROFILE/debug/train_progress.jsonl" || true
python -m efficient_vlm_ad monitor-progress --output-dir "$PROFILE" --last 5 || true

## 8. Evaluate và benchmark

Ưu tiên dùng `best_model.pt`; nếu file đó chưa có thì dùng `finetune_best.pt`.

In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
CONFIG="$QUESTION_GPA_CONFIG"
PROFILE="$QUESTION_GPA_PROFILE"
CHECKPOINT="$PROFILE/checkpoints/best_model.pt"
if [ ! -f "$CHECKPOINT" ]; then
  CHECKPOINT="$PROFILE/checkpoints/finetune_best.pt"
fi
if [ ! -f "$CHECKPOINT" ]; then
  echo "Missing evaluation checkpoint: $CHECKPOINT"
  exit 1
fi
METEOR_FLAG=""
if [ "${EVAL_WITH_METEOR:-0}" = "1" ]; then METEOR_FLAG="--with-meteor"; fi
python -m efficient_vlm_ad evaluate --config "$CONFIG" --checkpoint "$CHECKPOINT" $METEOR_FLAG
python -m efficient_vlm_ad benchmark --config "$CONFIG" --checkpoint "$CHECKPOINT"

## 9. Xem artifacts cuối cùng

Checkpoint, metrics và benchmark nằm trong output profile của safe question-GPA run.

In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
PROFILE="$QUESTION_GPA_PROFILE"
ls -lh "$PROFILE/checkpoints"
cat "$PROFILE/metrics.json"
cat "$PROFILE/benchmark.json"